# Notebook 1 - Data Processing 
Processes raw Optiver CSVs into model-ready features. 

## Cell 1 - Importing Packages 

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import glob
import re
import os
import json
warnings.filterwarnings('ignore')

## Cell 2 — Load Raw CSVs + WAP + Spread 
Reads each stock CSV, computes WAP and BidAskSpread, saves as individual parquet files to avoid memory crashes.


In [ ]:
# Find all stock CSV files
files = glob.glob("individual_book_train/stock_*.csv")
print(f"Found {len(files)} stock files")

# Create output folder for individual stock parquets
os.makedirs("processed_stocks", exist_ok=True)

for i, file in enumerate(files):

    df = pd.read_csv(file)

    stock_id = int(re.search(r"\d+", os.path.basename(file)).group())
    df["stock_id"] = stock_id

    df["wap"] = (
        (df["bid_price1"] * df["ask_size1"] + df["ask_price1"] * df["bid_size1"])
        / (df["bid_size1"] + df["ask_size1"])
    )

    df["bid_ask_spread"] = df["ask_price1"] / df["bid_price1"] - 1

    float_cols = df.select_dtypes(include="float64").columns
    df[float_cols] = df[float_cols].astype(np.float32)

    out_path = f"processed_stocks/stock_{stock_id}.parquet"
    df.to_parquet(out_path, index=False)

    del df

    if (i + 1) % 10 == 0:
        print(f"  Processed {i+1}/{len(files)} files...")

print(f"\nDone! Saved {len(files)} parquet files to processed_stocks/")

sample = pd.read_parquet("processed_stocks/stock_0.parquet")
print(f"\nSample stock_0 shape: {sample.shape}")
print(f"Columns: {list(sample.columns)}")
print(f"Missing values:\n{sample.isnull().sum()}")
print(f"\nseconds_in_bucket range: {sample['seconds_in_bucket'].min()} to {sample['seconds_in_bucket'].max()}")
print(f"\nWAP and spread summary:")
print(sample[["wap", "bid_ask_spread"]].describe())
sample.head(20)

Found 112 stock files
  Processed 10/112 files...
  Processed 20/112 files...
  Processed 30/112 files...
  Processed 40/112 files...
  Processed 50/112 files...
  Processed 60/112 files...
  Processed 70/112 files...
  Processed 80/112 files...
  Processed 90/112 files...
  Processed 100/112 files...
  Processed 110/112 files...

Done! Saved 112 parquet files to processed_stocks/

Sample stock_0 shape: (917553, 13)
Columns: ['time_id', 'seconds_in_bucket', 'bid_price1', 'ask_price1', 'bid_price2', 'ask_price2', 'bid_size1', 'ask_size1', 'bid_size2', 'ask_size2', 'stock_id', 'wap', 'bid_ask_spread']
Missing values:
time_id              0
seconds_in_bucket    0
bid_price1           0
ask_price1           0
bid_price2           0
ask_price2           0
bid_size1            0
ask_size1            0
bid_size2            0
ask_size2            0
stock_id             0
wap                  0
bid_ask_spread       0
dtype: int64

seconds_in_bucket range: 0 to 599

WAP and spread summary:
     

,time_id,seconds_in_bucket,bid_price1,ask_price1,bid_price2,ask_price2,bid_size1,ask_size1,bid_size2,ask_size2,stock_id,wap,bid_ask_spread
0,5,0,1.001422,1.002301,1.001370,1.002353,3,226,2,100,0,1.001434,0.000878
1,5,1,1.001422,1.002301,1.001370,1.002353,3,100,2,100,0,1.001448,0.000878
2,5,5,1.001422,1.002301,1.001370,1.002405,3,100,2,100,0,1.001448,0.000878
3,5,6,1.001422,1.002301,1.001370,1.002405,3,126,2,100,0,1.001443,0.000878
4,5,7,1.001422,1.002301,1.001370,1.002405,3,126,2,100,0,1.001443,0.000878
5,5,11,1.001422,1.002301,1.001370,1.002405,3,100,2,100,0,1.001448,0.000878
6,5,12,1.001422,1.002301,1.001370,1.002405,3,126,2,100,0,1.001443,0.000878
7,5,14,1.001422,1.002301,1.001370,1.002405,3,126,2,100,0,1.001443,0.000878
8,5,15,1.001422,1.002301,1.001370,1.002405,3,126,2,100,0,1.001443,0.000878
9,5,16,1.001422,1.002301,1.001370,1.002405,3,126,2,100,0,1.001443,0.000878


## Cell 3 — Reconstruct Full Grid + Forward Fill + Features 
- Expands each time_id to a full 600-second grid
- Forward-fills missing seconds (unchanged order book)
- Recomputes WAP and spread from filled values
- Computes log returns, 30s buckets, RV, and target_rv


In [ ]:
bucket_size = 30  

def realised_volatility(log_returns):
    """RV = sqrt(sum of squared log returns)."""
    return np.sqrt(np.nansum(log_returns ** 2))

def process_stock(stock_id):

    df = pd.read_parquet(f"processed_stocks/stock_{stock_id}.parquet")
    df = df.sort_values(["time_id", "seconds_in_bucket"]).reset_index(drop=True)

    
    all_time_ids = df["time_id"].unique()

    full_index = pd.MultiIndex.from_product(
        [all_time_ids, np.arange(0, 600)],
        names=["time_id", "seconds_in_bucket"]
    )
    df = (
        df.set_index(["time_id", "seconds_in_bucket"])
        .reindex(full_index)       
        .reset_index()
    )
    df["stock_id"] = stock_id

    
    book_cols = [
        "bid_price1", "ask_price1", "bid_price2", "ask_price2",
        "bid_size1",  "ask_size1",  "bid_size2",  "ask_size2",
    ]
    df[book_cols] = (
        df.groupby("time_id")[book_cols]
        .transform(lambda x: x.ffill())
    )

    
    df = df.dropna(subset=book_cols).reset_index(drop=True)

    df["wap"] = (
        (df["bid_price1"] * df["ask_size1"] + df["ask_price1"] * df["bid_size1"])
        / (df["bid_size1"] + df["ask_size1"])
    )
    df["bid_ask_spread"] = df["ask_price1"] / df["bid_price1"] - 1

    df["log_return"] = (
        df.groupby("time_id")["wap"]
        .transform(lambda x: np.log(x).diff().fillna(0))
    )

    df["bucket"] = (df["seconds_in_bucket"] // bucket_size).astype(int)

    agg = (
        df.groupby(["stock_id", "time_id", "bucket"])
        .agg(
            wap_last        = ("wap",           "last"),
            wap_mean        = ("wap",           "mean"),
            spread_mean     = ("bid_ask_spread","mean"),
            spread_max      = ("bid_ask_spread","max"),
            log_return_sum  = ("log_return",    "sum"),
            log_return_std  = ("log_return",    "std"),
            n_obs           = ("wap",           "count"),  # always 30 now
        )
        .reset_index()
    )

    rv = (
        df.groupby(["stock_id", "time_id", "bucket"])["log_return"]
        .apply(realised_volatility)
        .reset_index()
        .rename(columns={"log_return": "realised_volatility"})
    )

    features = agg.merge(rv, on=["stock_id", "time_id", "bucket"], how="left")

    features = features.sort_values(["stock_id", "time_id", "bucket"]).reset_index(drop=True)
    features["target_rv"] = (
        features.groupby(["stock_id", "time_id"])["realised_volatility"]
        .shift(-1)
    )
    features = features.dropna(subset=["target_rv"]).reset_index(drop=True)

    float_cols = features.select_dtypes(include="float64").columns
    features[float_cols] = features[float_cols].astype(np.float32)

    return features


os.makedirs("features_stocks", exist_ok=True)

stock_ids = sorted([
    int(re.search(r"\d+", f).group())
    for f in glob.glob("processed_stocks/stock_*.parquet")
])

for i, stock_id in enumerate(stock_ids):
    result = process_stock(stock_id)
    result.to_parquet(f"features_stocks/stock_{stock_id}.parquet", index=False)
    del result

    if (i + 1) % 10 == 0:
        print(f"  Processed {i+1}/{len(stock_ids)} stocks...")

print(f"\nDone! Saved {len(stock_ids)} feature files to features_stocks/")

sample = pd.read_parquet('features_stocks/stock_0.parquet')
print(f'\nSample stock_0 shape: {sample.shape}')
print(f'n_obs per bucket (expect 30):')
print(sample['n_obs'].value_counts())
sample.head()

  Processed 10/112 stocks...
  Processed 20/112 stocks...
  Processed 30/112 stocks...
  Processed 40/112 stocks...
  Processed 50/112 stocks...
  Processed 60/112 stocks...
  Processed 70/112 stocks...
  Processed 80/112 stocks...
  Processed 90/112 stocks...
  Processed 100/112 stocks...
  Processed 110/112 stocks...

Done! Saved 112 feature files to features_stocks/

Sample stock_0 shape: (72770, 12)
n_obs per bucket (expect 30):
n_obs
30    72770
Name: count, dtype: int64


,stock_id,time_id,bucket,wap_last,wap_mean,spread_mean,spread_max,log_return_sum,log_return_std,n_obs,realised_volatility,target_rv
0,0,5,0,1.002530,1.001674,0.000979,0.001394,0.001094,0.000193,30,0.001057,0.001266
1,0,5,1,1.003624,1.002671,0.000903,0.001135,0.001090,0.000232,30,0.001266,0.001090
2,0,5,2,1.004103,1.003549,0.000883,0.000928,0.000477,0.000202,30,0.001090,0.000367
3,0,5,3,1.003797,1.004068,0.000871,0.000928,-0.000305,0.000067,30,0.000367,0.001339
4,0,5,4,1.003454,1.004353,0.000725,0.000979,-0.000341,0.000248,30,0.001339,0.001478


## Cell 4 — Setup: Sample 60 Stocks, Seen/Unseen Split
- Randomly selects 60 stocks from 112
- Splits into 40 seen (model development) and 20 unseen (final evaluation)
- Loads only seen stocks into memory

In [ ]:
 
seed = 3888
np.random.seed(seed)


all_stock_ids = sorted([
    int(re.search(r"\d+", f).group())
    for f in glob.glob("features_stocks/stock_*.parquet")
])
print(f"Total stocks available: {len(all_stock_ids)}")


Keep_N = 60
sampled_stocks = sorted(np.random.choice(all_stock_ids, size=Keep_N, replace=False).tolist())
print(f"Stocks kept: {len(sampled_stocks)}")


N_UNSEEN = 20
unseen_stocks = sorted(np.random.choice(sampled_stocks, size=N_UNSEEN, replace=False).tolist())
seen_stocks   = sorted([s for s in sampled_stocks if s not in unseen_stocks])

print(f"\nSeen stocks   ({len(seen_stocks)}):   {seen_stocks}")
print(f"Unseen stocks ({len(unseen_stocks)}): {unseen_stocks}")


seen_dfs = []
for stock_id in seen_stocks:
    df = pd.read_parquet(f"features_stocks/stock_{stock_id}.parquet")
    seen_dfs.append(df)

seen_data = pd.concat(seen_dfs, ignore_index=True)
del seen_dfs


seen_data["bucket"] = seen_data["bucket"] + 1

print(f"\nSeen data shape: {seen_data.shape}")
print(f"Buckets: {sorted(seen_data['bucket'].unique())}")
print(f"\nSeen data preview:")
seen_data.head()

Total stocks available: 112
Stocks kept: 60

Seen stocks   (40):   [5, 7, 10, 13, 14, 16, 18, 19, 20, 27, 29, 37, 43, 44, 46, 51, 61, 62, 63, 64, 68, 69, 70, 72, 75, 77, 78, 80, 82, 83, 90, 98, 100, 102, 103, 104, 115, 118, 120, 126]
Unseen stocks (20): [0, 17, 22, 28, 30, 34, 35, 38, 40, 48, 60, 74, 81, 84, 87, 89, 93, 95, 105, 111]

Seen data shape: (2910553, 12)
Buckets: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19)]

Seen data preview:


,stock_id,time_id,bucket,wap_last,wap_mean,spread_mean,spread_max,log_return_sum,log_return_std,n_obs,realised_volatility,target_rv
0,5,5,1,1.001802,1.001516,0.001219,0.002005,0.000479,0.000123,30,0.000670,0.001268
1,5,5,2,1.001797,1.001448,0.001110,0.001327,-0.000005,0.000235,30,0.001268,0.001828
2,5,5,3,1.001324,1.001776,0.001306,0.001598,-0.000473,0.000339,30,0.001828,0.001479
3,5,5,4,1.000361,1.000403,0.001481,0.001598,-0.000962,0.000273,30,0.001479,0.001766
4,5,5,5,1.000747,1.000877,0.001462,0.001652,0.000386,0.000328,30,0.001766,0.001483


## Cell 5 — Save Everything for Modelling Notebook

In [ ]:


seen_data.to_parquet('seen_data.parquet', index=False)


unseen_dfs = []
for stock_id in unseen_stocks:
    df = pd.read_parquet(f'features_stocks/stock_{stock_id}.parquet')
    unseen_dfs.append(df)

unseen_data = pd.concat(unseen_dfs, ignore_index=True)
del unseen_dfs


unseen_data['bucket'] = unseen_data['bucket'] + 1

unseen_data.to_parquet('unseen_data.parquet', index=False)


with open('stock_split.json', 'w') as f:
    json.dump({
        'seen_stocks':    seen_stocks,
        'unseen_stocks':  unseen_stocks,
        'sampled_stocks': sampled_stocks
    }, f)

print('Saved:')
print('  seen_data.parquet   — feature table for seen stocks')
print('  unseen_data.parquet — feature table for unseen stocks')
print('  stock_split.json    — seen/unseen stock ID lists')
print(f'\nseen_data shape:   {seen_data.shape}')
print(f'unseen_data shape: {unseen_data.shape}')
print(f'Seen stocks:   {len(seen_stocks)}')
print(f'Unseen stocks: {len(unseen_stocks)}')
print(f'\nUnseen stock IDs: {sorted(unseen_data["stock_id"].unique().tolist())}')
print(f'Unseen buckets:   {sorted(unseen_data["bucket"].unique())}')

Saved:
  seen_data.parquet   — feature table for seen stocks
  unseen_data.parquet — feature table for unseen stocks
  stock_split.json    — seen/unseen stock ID lists

seen_data shape:   (2910553, 12)
unseen_data shape: (1455115, 12)
Seen stocks:   40
Unseen stocks: 20

Unseen stock IDs: [0, 17, 22, 28, 30, 34, 35, 38, 40, 48, 60, 74, 81, 84, 87, 89, 93, 95, 105, 111]
Unseen buckets:   [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19)]


In [ ]:
import glob
print(glob.glob("processed_stocks/stock_*.parquet"))  

['processed_stocks/stock_94.parquet', 'processed_stocks/stock_84.parquet', 'processed_stocks/stock_103.parquet', 'processed_stocks/stock_113.parquet', 'processed_stocks/stock_0.parquet', 'processed_stocks/stock_70.parquet', 'processed_stocks/stock_60.parquet', 'processed_stocks/stock_56.parquet', 'processed_stocks/stock_46.parquet', 'processed_stocks/stock_34.parquet', 'processed_stocks/stock_9.parquet', 'processed_stocks/stock_69.parquet', 'processed_stocks/stock_125.parquet', 'processed_stocks/stock_47.parquet', 'processed_stocks/stock_35.parquet', 'processed_stocks/stock_78.parquet', 'processed_stocks/stock_68.parquet', 'processed_stocks/stock_8.parquet', 'processed_stocks/stock_124.parquet', 'processed_stocks/stock_112.parquet', 'processed_stocks/stock_102.parquet', 'processed_stocks/stock_85.parquet', 'processed_stocks/stock_95.parquet', 'processed_stocks/stock_13.parquet', 'processed_stocks/stock_61.parquet', 'processed_stocks/stock_1.parquet', 'processed_stocks/stock_119.parquet